# Week 2-1 · 티켓 판단을 structured output으로 제한하기

## 시나리오
자유 형식 문의를 graph가 안정적으로 사용할 수 있는 category·priority·reason 구조로 변환하고 잘못된 출력은 거부합니다.

## 학습 목표
- `Literal`과 Pydantic으로 허용 값 집합을 선언한다.
- 모델 응답과 같은 dict를 `model_validate`로 파싱한다.
- 잘못된 category가 다음 node로 전파되지 않게 한다.

## 직접 조립
완성된 `weekX.app` 함수를 가져오지 않습니다. 아래 코드에서 작은 fixture와 핵심 객체·함수·연결을 직접 만듭니다.

### 1단계 · 출력 schema 선언

In [ ]:
# 실행 순서: 1단계 · 출력 schema 선언에서 TicketClassification을(를) 먼저 구성합니다.
# 관찰 포인트: 이 셀의 출력이 다음 단계에서 사용할 입력 계약을 충족하는지 확인합니다 — 1단계 · 출력 schema 선언.
from typing import Literal
from pydantic import BaseModel, ValidationError

# 자유 형식 판단을 허용된 category와 priority 값으로 제한합니다.
class TicketClassification(BaseModel):
    category: Literal["billing", "access", "technical", "other"]
    priority: Literal["normal", "urgent"]
    reason: str

practice_model_output = {
    "category": "billing",
    "priority": "normal",
    "reason": "중복 결제 표현이 있음",
}
classification = TicketClassification.model_validate(practice_model_output)
classification.model_dump()

### 2단계 · invalid output 관찰

In [ ]:
# 실행 순서: 2단계 · invalid output 관찰에서 fixture와 assertion을(를) 먼저 구성합니다.
# 관찰 포인트: 이 셀의 출력이 다음 단계에서 사용할 입력 계약을 충족하는지 확인합니다 — 2단계 · invalid output 관찰.
try:
    TicketClassification.model_validate({
        "category": "refund-now",
        "priority": "critical",
        "reason": "schema 밖 값",
    })
except ValidationError as error:
    practice_validation_errors = error.errors()

[(item["loc"], item["type"]) for item in practice_validation_errors]

### 3단계 · downstream 계약

In [ ]:
# 실행 순서: 3단계 · downstream 계약에서 fixture와 assertion을(를) 먼저 구성합니다.
# 관찰 포인트: 이 셀의 출력이 다음 단계에서 사용할 입력 계약을 충족하는지 확인합니다 — 3단계 · downstream 계약.
assert classification.category == "billing"
assert classification.priority == "normal"
assert {item["loc"][0] for item in practice_validation_errors} == {"category", "priority"}
{"route_input": classification.category, "accepted": True, "invalid_field_count": len(practice_validation_errors)}

## 중간 결과
각 코드 셀의 출력에서 입력이 어떤 상태로 변했는지 확인합니다. 마지막 `assert`는 눈으로 본 결과를 실행 가능한 계약으로 고정합니다.

## 실패 경계
schema 밖의 값은 임의의 fallback category로 바꾸지 않고 validation failure로 드러냅니다. 분류는 환불이나 계정 변경을 실행하지 않습니다.

## 실제 app 연결
Week 2 live app에서는 Chat model의 `with_structured_output(TicketClassification)`이 같은 schema를 사용합니다. 여기서는 모델 호출 없이 parsing 계약을 직접 검증합니다.

### 확장 과제
fixture의 문장이나 임계값을 하나 바꾸고, 어느 중간 결과와 assertion이 달라지는지 기록하세요.

## 다음 Notebook 연결
다음 `02_metadata_filtered_retrieval.ipynb`에서는 분류 category를 검색 후보를 제한하는 metadata filter로 사용합니다.